# Week 3, day 2 — Graded 01 SOLUTIONS: Foundations   (tier 1 of 4)

Executed in the lab image (pandas 3.0.5) against the real files in `../data/`.
Every quoted number is what it actually printed.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Graded 01 — Foundations. Run this once.
import pandas as pd

orders = pd.read_csv("../data/orders_long.csv")
cust = pd.read_csv("../data/customers_messy.csv")

print("orders:", orders.shape, "| customers:", cust.shape)

TIER 1 — one operation per question

### Question 1

`1093` rows, 11 columns.

The first thing to do with any file. If this number is not what you
expected, stop here -- nothing computed afterwards will be right.

In [ ]:
print("rows:", len(orders))
print("columns:", list(orders.columns))

### Question 2

`Office Supplies 606`, `Technology 324`, `Furniture 163`.

They sum to 1,093, so every row has a category and none is missing.
Checking that the counts add up to the row count is free and catches a
surprising amount.

In [ ]:
print(orders["Category"].value_counts().to_string())

### Question 3

**`29`** duplicate rows of `440`.

`duplicated()` marks the second and later occurrences, so 29 is the number
you would delete, not the number of rows involved. `keep=False` would say 58.

In [ ]:
print("duplicate rows:", cust.duplicated().sum(), "of", len(cust))

### Question 4

Ontario `466003.99`, West `346344.89`, Atlantic `236399.66`, Prarie `233047.01`, then Quebec, Yukon, NWT, Nunavut `15890.71`.

Note `Prarie` -- the source spells it without the second `i`. Filtering for
the correct English spelling returns nothing, with no error.

In [ ]:
print(orders.groupby("Region")["Sales"].sum().round(2)
            .sort_values(ascending=False).to_string())

### Question 5

`32` rows, index levels `['Region', 'Year']`, index type **`MultiIndex`**.

Grouping by two columns produces a two-level index without you asking for
one. 8 regions x 4 years = 32, so every combination is present.

The level order is the order you passed to `groupby`, and it decides which
selections are cheap later.

In [ ]:
g = orders.groupby(["Region", "Year"])["Sales"].sum()
print("rows:", len(g))
print("index levels:", g.index.names)
print("index type:", type(g.index).__name__)

### Question 6

`unstack()` -> shape **`(8, 4)`**, columns `[2009, 2010, 2011, 2012]`.

The innermost level became the columns. 32 rows became 32 cells -- same
data, rectangular.

The column labels are **integers**, not strings, because `Year` is `int64`
in the file. `wide["2009"]` raises; `wide[2009]` works.

In [ ]:
wide = orders.groupby(["Region", "Year"])["Sales"].sum().unstack()
print("shape:", wide.shape)
print("columns:", list(wide.columns))
print()
print(wide.round(2).to_string())

### Question 7

`small 586`, `medium 411`, `large 96` -> `1093` binned, none left out.

The counts add to the row count because the top edge is `inf`. Had it been
a finite number, any order above it would have become `NaN` and dropped out
of the tally silently -- see graded 02 Q10 and graded 03 Q2.

More than half the orders are under 500.

In [ ]:
band = pd.cut(orders["Sales"], bins=[0, 500, 5000, float("inf")],
              labels=["small", "medium", "large"])
print(band.value_counts().to_string())
print()
print("binned:", band.notna().sum(), "of", len(orders))

### Question 8

Missing `Province` goes from **`17`** to **`50`**.

The 17 already-missing rows were the `N/A` markers, which `read_csv`
converts to `NaN` automatically -- `N/A` is on its default list. `-` and `?`
are not, so they arrived as ordinary text and needed replacing.

One file, three markers meaning the same thing, and one of them handled for
you without mention.

In [ ]:
import numpy as np
print("missing before:", cust["Province"].isna().sum())
fixed = cust["Province"].replace(["-", "?"], np.nan)
print("missing after: ", fixed.isna().sum())

### Question 9

The `Region` x `Category` table sums to `1605576.22`, matching `orders["Sales"].sum()`.

`aggfunc="sum"` is passed explicitly, which is the point of the question.
The default is `mean`, and a table of means looks exactly like a table of
totals -- same shape, same formatting, numbers that are plausible either way.

The reconciliation check only works for additive aggregates. It is worth
running on every summary table you publish.

In [ ]:
pt = orders.pivot_table(index="Region", columns="Category",
                        values="Sales", aggfunc="sum")
print(pt.round(2).to_string())
print()
print("grand total:", round(pt.sum().sum(), 2))
print("raw total:  ", round(orders["Sales"].sum(), 2))

### Question 10

`pivot()` -> **raises** `ValueError: Index contains duplicate entries, cannot reshape`.

`pivot()` only rearranges: it needs exactly one row per index/column pair
and refuses when there are more. Here each Region/Category pair has dozens
of orders.

`pivot_table()` handled it in Q9 because aggregating is precisely what it
adds -- and that is why it needs an `aggfunc`, and why leaving the aggfunc
out gives you a table of averages.

The rule: **`pivot` for a reshape, `pivot_table` when a cell may hold more
than one record.** If you are unsure which you have, `groupby(...).size().max()`
tells you.

In [ ]:
print("rows per Region/Category pair, largest:")
print(orders.groupby(["Region", "Category"]).size().nlargest(3).to_string())
print()
print(orders.pivot(index="Region", columns="Category", values="Sales"))